# 20 Newsgroups Row-ID Prefilter QOS Adapter (Colab)

This notebook applies `mod30`, `mod210`, and `mod2310` wheel prefilters to a real text-classification dataset: `20newsgroups`.

The workflow is adapter-level and non-invasive:

```text
20news text dataset
    → TF-IDF matrix X, labels y
    → row/sample IDs
    → wheel prefilter
    → X_filtered, y_filtered
    → downstream classifier sanity check
```

Guardrail: this notebook reports retained sample counts, fit-time proxy, and classifier behavior. It does **not** claim wheel filtering improves accuracy or proves quantum speedup.

In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS
from pre_oracle_filter import pre_oracle_candidates_by_name

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.utils.validation import check_is_fitted

## 1. Load 20 Newsgroups subset

We use four categories to keep runtime fast and interpretation simple. This still gives a real sparse text dataset with meaningful labels.

In [ ]:
RANDOM_STATE = 9423

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = dataset.data
y = np.array(dataset.target)
target_names = dataset.target_names

len(texts), len(y), target_names

## 2. Vectorize text with TF-IDF

This creates a sparse matrix similar in spirit to real text-feature workflows in `real_datasets/`.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=12000,
    min_df=2,
    stop_words="english",
)

X = vectorizer.fit_transform(texts)

X.shape, y.shape, X.nnz

## 3. Row-ID wheel prefilter adapter

This is the safe insertion point for real-dataset workflows: filter row IDs, then slice `X` and `y` before downstream processing.

In [ ]:
def filter_dataset_by_row_ids(X, y, wheel_name="mod30"):
    """Filter dataset rows using wheel-admissible sample indices."""
    row_ids = np.arange(X.shape[0])
    filtered_ids = np.array(pre_oracle_candidates_by_name(row_ids, wheel_name), dtype=int)
    return X[filtered_ids], y[filtered_ids], filtered_ids


def class_distribution(y_values, n_classes):
    counts = np.bincount(y_values, minlength=n_classes)
    fractions = counts / counts.sum()
    return counts, fractions


filter_dataset_by_row_ids(X, y, "mod30")[0].shape

## 4. Retained samples and class balance

The first result checks whether row-ID filtering follows the wheel density and how class balance shifts.

In [ ]:
n_classes = len(target_names)
baseline_counts, baseline_fracs = class_distribution(y, n_classes)

retained_rows = []
for wheel_name, wheel in STANDARD_WHEELS.items():
    Xf, yf, ids = filter_dataset_by_row_ids(X, y, wheel_name)
    counts, fracs = class_distribution(yf, n_classes)
    balance_l1_shift = float(np.sum(np.abs(fracs - baseline_fracs)))
    retained_rows.append({
        "subset": wheel_name,
        "modulus": wheel.modulus,
        "residues": wheel.residue_count,
        "n_original": X.shape[0],
        "n_retained": Xf.shape[0],
        "retained_fraction": Xf.shape[0] / X.shape[0],
        "reduction_fraction": 1 - Xf.shape[0] / X.shape[0],
        "class_balance_l1_shift": balance_l1_shift,
    })

retained = pd.DataFrame(retained_rows)
retained

## 5. Sparse LinearSVC sanity benchmark

This benchmark checks downstream behavior after filtering. It uses stratified train/test splits after filtering so each subset is evaluated fairly on its own retained data.

In [ ]:
def evaluate_subset(X_subset, y_subset, label):
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset,
        y_subset,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y_subset,
    )

    clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
    t0 = time.perf_counter()
    clf.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0

    pred = clf.predict(X_test)
    return {
        "subset": label,
        "n_samples": X_subset.shape[0],
        "n_features": X_subset.shape[1],
        "nnz": X_subset.nnz,
        "fit_time_seconds": fit_time,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    }


eval_rows = [evaluate_subset(X, y, "baseline_all_rows")]
for wheel_name in STANDARD_WHEELS:
    Xf, yf, ids = filter_dataset_by_row_ids(X, y, wheel_name)
    eval_rows.append(evaluate_subset(Xf, yf, wheel_name))

eval_df = pd.DataFrame(eval_rows)
eval_df

## 6. Save tables and figures

Outputs are saved to `data/` and `figures/` for README/paper reuse.

In [ ]:
fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

retained_csv = data_dir / "20news_row_id_prefilter_retained_samples.csv"
eval_csv = data_dir / "20news_row_id_prefilter_svc_sanity.csv"
retained.to_csv(retained_csv, index=False)
eval_df.to_csv(eval_csv, index=False)

# Figure 1: retained sample fraction
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(retained["subset"], retained["retained_fraction"])
ax.set_title("20news Row-ID Prefilter: Retained Sample Fraction")
ax.set_xlabel("Wheel filter")
ax.set_ylabel("Retained sample fraction")
ax.set_ylim(0, 0.32)
ax.grid(True, axis="y", alpha=0.35)
for i, value in enumerate(retained["retained_fraction"]):
    ax.text(i, value + 0.01, f"{value:.1%}", ha="center")
fig.tight_layout()
retained_svg = fig_dir / "20news_row_id_prefilter_retained_fraction.svg"
retained_png = fig_dir / "20news_row_id_prefilter_retained_fraction.png"
fig.savefig(retained_svg)
fig.savefig(retained_png, dpi=200)
plt.show()

# Figure 2: balanced accuracy
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.plot(eval_df["subset"], eval_df["balanced_accuracy"], marker="o")
ax.set_title("20news LinearSVC Sanity Check After Row-ID Prefiltering")
ax.set_xlabel("Subset")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
plt.xticks(rotation=20)
fig.tight_layout()
acc_svg = fig_dir / "20news_row_id_prefilter_balanced_accuracy.svg"
acc_png = fig_dir / "20news_row_id_prefilter_balanced_accuracy.png"
fig.savefig(acc_svg)
fig.savefig(acc_png, dpi=200)
plt.show()

# Figure 3: fit time
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.bar(eval_df["subset"], eval_df["fit_time_seconds"])
ax.set_title("20news LinearSVC Fit-Time Proxy After Row-ID Prefiltering")
ax.set_xlabel("Subset")
ax.set_ylabel("Fit time (seconds)")
ax.grid(True, axis="y", alpha=0.35)
plt.xticks(rotation=20)
fig.tight_layout()
time_svg = fig_dir / "20news_row_id_prefilter_fit_time.svg"
time_png = fig_dir / "20news_row_id_prefilter_fit_time.png"
fig.savefig(time_svg)
fig.savefig(time_png, dpi=200)
plt.show()

retained_csv, eval_csv, retained_svg, acc_svg, time_svg

## 7. Adapter notes for `real_datasets/20news_svm.py`

A non-invasive integration path is to insert the row-ID filter after `X, y` are created and before model/QOS calls:

```python
from pre_oracle_filter import pre_oracle_candidates_by_name

row_ids = range(X.shape[0])
filtered_ids = pre_oracle_candidates_by_name(row_ids, "mod30")
X = X[filtered_ids]
y = y[filtered_ids]
```

This keeps the wheel layer optional and external to `qos.py`, `qos_sampling.py`, and `qsvt.py`.

## 8. Interpretation

- The row-ID filter reproduces wheel-density reductions on a real sparse text dataset.
- The classifier sanity check reports downstream behavior after large sample reduction.
- Fit time acts as a simple classical workload proxy.
- The next step is either a direct wrapper around `real_datasets/20news_svm.py` or a paper figure pack summarizing Notebooks 1--3.

In [ ]:
# Optional: download outputs in Colab
from google.colab import files

for path in [
    "data/20news_row_id_prefilter_retained_samples.csv",
    "data/20news_row_id_prefilter_svc_sanity.csv",
    "figures/20news_row_id_prefilter_retained_fraction.svg",
    "figures/20news_row_id_prefilter_balanced_accuracy.svg",
    "figures/20news_row_id_prefilter_fit_time.svg",
]:
    files.download(path)